# 04.04 · Pretraining, SFT, RLHF, reasoning RL

**Chapter:** Transformers & How LLMs Work · **Milestone:** M1

## Goals
- [ ] The full pipeline: pretraining → supervised fine-tuning (SFT) → preference tuning → RL for reasoning
- [ ] Estimate training compute and data budgets (scaling laws, Chinchilla)
- [ ] See what SFT data and chat templates look like
- [ ] Understand the DPO loss and RL with verifiable rewards at toy scale

## Resources
- [Karpathy – Deep Dive into LLMs like ChatGPT](https://www.youtube.com/watch?v=7xTGNNLPyMI) — free (3.5 h; the best overview that exists)
- [Stanford CS336 – Language Modeling from Scratch](https://stanford-cs336.github.io/) — free
- [Chinchilla paper: Training Compute-Optimal LLMs](https://arxiv.org/abs/2203.15556) · [DPO paper](https://arxiv.org/abs/2305.18290) · [DeepSeek-R1](https://arxiv.org/abs/2501.12948) — free

## 1. The pipeline

| Stage | Data | Objective | Result |
|---|---|---|---|
| **Pretraining** | trillions of tokens of web, code, books | next-token prediction (what you did in 04.03) | *base model*: autocompletes documents, knows a lot, doesn't follow instructions |
| **SFT** (instruction tuning) | 10k–1M curated (prompt, ideal response) conversations | next-token loss on the *assistant* tokens only | *chat model*: follows the format of a helpful assistant |
| **Preference tuning** (RLHF / DPO) | pairs: response A preferred over B (by humans or AI) | increase P(preferred) relative to P(rejected), stay close to SFT model | more helpful, honest, harmless; better style |
| **RL with verifiable rewards** | math/code problems with checkable answers | reward = answer correct / tests pass | *reasoning model*: long chains of thought, self-correction |

Plus: data filtering & deduplication, tokenizer training, safety training, tool-use training, distillation from bigger models.

## 2. Compute: the 6ND rule and Chinchilla
Training FLOPs ≈ **6 × N (parameters) × D (tokens)** (2 for the forward pass, 4 for backward).
Chinchilla (2022): for a fixed compute budget, the loss-optimal ratio is **≈ 20 tokens per parameter**.
Modern models train *far* past that (e.g. 15T+ tokens for 8B models) because a smaller model trained longer is cheaper to **serve**.

In [ ]:
def training_cost(params, tokens, gpu_flops=989e12, mfu=0.4, usd_per_gpu_hour=2.5):
    """gpu_flops: H100 dense bf16 peak. mfu: fraction of peak you actually achieve."""
    flops = 6 * params * tokens
    gpu_hours = flops / (gpu_flops * mfu) / 3600
    return flops, gpu_hours, gpu_hours * usd_per_gpu_hour

for name, N, D in [
    ("your 04.03 tiny GPT", 1.8e6, 2000 * 32 * 128),
    ("GPT-2 small (124M), Chinchilla-optimal", 124e6, 20 * 124e6),
    ("8B model, 15T tokens", 8e9, 15e12),
    ("70B model, 15T tokens", 70e9, 15e12),
]:
    flops, hours, usd = training_cost(N, D)
    print(f"{name:42} {flops:9.2e} FLOPs | {hours:12,.1f} H100-hours | ~${usd:,.0f}")

(Illustrative: assumes 40% utilisation at \$2.5/GPU-hour; real runs add failed experiments, data work and people.)
This is why AI engineers mostly **use and adapt** models rather than pretrain them.

## 3. SFT data and chat templates
A chat is serialized to one token sequence with special tokens marking roles. Each model family has its own template —
using the wrong one silently degrades quality. Here's Qwen3's:

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
conversation = [
    {"role": "system", "content": "You are a concise assistant."},
    {"role": "user", "content": "What is SFT?"},
    {"role": "assistant", "content": "Supervised fine-tuning on example conversations."},
]
print(tok.apply_chat_template(conversation, tokenize=False))

Notice the empty `<think></think>` in the assistant turn: Qwen3 is a hybrid reasoning model and its template marks the thinking
section even when it's empty.

During SFT, the loss is usually computed only on the assistant's tokens (prompt tokens are "masked" with label `-100`),
so the model learns to *answer*, not to imitate users.

## 4. Preference tuning: the DPO loss
Given a prompt with chosen response $y_w$ and rejected $y_l$, policy $\pi_\theta$ and frozen reference $\pi_{ref}$ (the SFT model):

$$\mathcal{L}_{DPO} = -\log\sigma\left(\beta\left[\log\frac{\pi_\theta(y_w)}{\pi_{ref}(y_w)} - \log\frac{\pi_\theta(y_l)}{\pi_{ref}(y_l)}\right]\right)$$

Push up the chosen response's likelihood *relative to the reference* more than the rejected one's. $\beta$ controls how far
the policy may drift. No reward model, no RL loop — just a classification-style loss on sequence log-probs.

In [ ]:
import torch
import torch.nn.functional as F

def dpo_loss(pi_chosen, pi_rejected, ref_chosen, ref_rejected, beta=0.1):
    """Inputs: summed log-probs of each full response under policy / reference."""
    margin = (pi_chosen - ref_chosen) - (pi_rejected - ref_rejected)
    return -F.logsigmoid(beta * margin).mean()

ref_c, ref_r = torch.tensor([-42.0]), torch.tensor([-40.0])  # reference slightly prefers the *rejected* answer
for pi_c, pi_r in [(-42.0, -40.0), (-38.0, -44.0), (-30.0, -60.0)]:
    print(f"policy chosen={pi_c}, rejected={pi_r} -> loss {dpo_loss(torch.tensor([pi_c]), torch.tensor([pi_r]), ref_c, ref_r):.4f}")

## 5. RL with verifiable rewards (toy)
Reasoning models (o-series, DeepSeek-R1, Qwen3 thinking…) are trained by sampling many attempts per problem, scoring them with
a **program** (is the final answer correct? do tests pass?), and reinforcing the better ones. **GRPO** normalises rewards within each
group of samples for the same prompt: advantage = (reward − group mean) / group std.

Toy version: a "policy" over 4 strategies for adding numbers, one of which is correct. REINFORCE with group-normalised advantages.

In [ ]:
torch.manual_seed(0)
strategies = ["a + b", "a - b", "a * b", "a + b + 1"]
logits = torch.zeros(4, requires_grad=True)
opt = torch.optim.Adam([logits], lr=0.1)

def reward(strategy_idx, a, b):
    return float(eval(strategies[strategy_idx]) == a + b)  # verifiable: we can check the answer

for step in range(60):
    a, b = torch.randint(0, 100, (2,)).tolist()
    probs = logits.softmax(-1)
    samples = torch.multinomial(probs, 8, replacement=True)  # a "group" of 8 attempts
    rewards = torch.tensor([reward(s, a, b) for s in samples])
    adv = (rewards - rewards.mean()) / (rewards.std() + 1e-6)  # GRPO-style group normalisation
    loss = -(adv * torch.log(probs[samples])).mean()
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 15 == 0 or step == 59:
        print(f"step {step:2}: " + "  ".join(f"{s}={p:.2f}" for s, p in zip(strategies, logits.softmax(-1).tolist())))

Real GRPO adds a KL penalty to the reference model and clipping (like PPO), and the "strategies" are entire generated chains of thought.
You'll run real GRPO in chapter 12.05.

## Exercises
1. Write a one-page explainer of how a chat model is made, in your own words, without looking at this notebook.
2. Using `training_cost`, how many tokens could you train a 124M model on for \$100? Is that Chinchilla-optimal?
3. Find the chat template of `HuggingFaceTB/SmolLM2-135M-Instruct` and of a Llama or Gemma model. What differs?
4. In the GRPO toy, what happens if all 8 samples get the same reward? Why is that a problem for very hard (or very easy) prompts?

In [ ]:
# your code here